# 04 — Evaluation

Comparación de los 3 experimentos de `03_modeling.ipynb`, discusión de los
límites del weak label, y registro del mejor modelo en el Model Registry de
MLflow.


In [ ]:
import sys
from pathlib import Path

SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

import mlflow
import mlflow_utils as MU

MU.set_tracking_uri()
experiment = mlflow.get_experiment_by_name("otdr_risk_detection")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
cols = [c for c in runs.columns if c.startswith("metrics.") or c == "tags.mlflow.runName"]
runs[cols].sort_values("tags.mlflow.runName")


## Métrica de negocio ad-hoc

Además de precision/recall/F1 contra el weak label, una validación
semi-supervisada más honesta: **de las rutas que el modelo marcó de alto
riesgo, ¿qué fracción tuvo después una medición real con `FaultStatus =
Detected`?** Esto usa el futuro como proxy de verdad, ya que no hay ground
truth curado (ver `00_business_understanding.ipynb`).


In [ ]:
PROCESSED_DIR = Path("../data/processed")
df = pd.read_parquet(PROCESSED_DIR / "features_by_route_timeline.parquet").sort_values("TestTime")

split_idx = int(len(df) * 0.7)
df_test = df.iloc[split_idx:].copy()

# TODO: usar las predicciones del modelo elegido (recalcular o pasar desde 03_modeling)
# df_test["predicted_at_risk"] = ...

# future_fault_rate = (
#     df_test[df_test["predicted_at_risk"] == 1]
#     .groupby("route_id")
#     .apply(lambda g: (g["FaultStatus"] == "Detected").any())
#     .mean()
# )
# print(f"Fracción de rutas de alto riesgo con falla real posterior: {future_fault_rate:.1%}")


## Discusión de límites del weak label

- El modelo supervisado puede estar simplemente re-aprendiendo la regla de
  3dB (`delta_loss_db` es, por construcción, el feature más correlacionado
  con la etiqueta) — un F1 alto no prueba generalización a fallas no vistas.
- `IsolationForest` no usa la etiqueta para entrenar, así que su comparación
  contra `FaultStatus=Detected` real es más informativa de la capacidad real
  de detectar anomalías, aunque su `contamination` es un hiperparámetro
  arbitrario que conviene barrer.
- La inspección manual de las anomalías top-N de IsolationForest (¿coinciden
  con rutas que el NOC reconocería como problemáticas?) es cualitativa y debe
  documentarse aquí tras revisión humana.


## Selección y registro del mejor modelo

In [ ]:
# TODO: completar con el run_id real del modelo elegido (ej. random_forest_supervised
# de 03_modeling.ipynb, si su F1/recall supera claramente a la regla y a IsolationForest).
BEST_RUN_ID = None
MODEL_NAME = "otdr_risk_classifier"

if BEST_RUN_ID:
    version = MU.register_best_model(BEST_RUN_ID, model_name=MODEL_NAME)
    MU.promote_model(MODEL_NAME, version, alias="production")
    print(f"[+] {MODEL_NAME} v{version} promovido a alias 'production'")
